Extract + Load: copy the raw csv files into DuckDb, untouched.

In [1]:
import sys
import duckdb

In [2]:
DB = "shopflow.duckdb"

In [3]:
TABLES = ["customers", "products", "orders", "order_items", "payments"]

In [4]:
con = duckdb.connect(DB)
con.execute("CREATE SCHEMA IF NOT EXISTS raw")

In [ ]:
# query = """select
#     payment_id,
#     order_id,
#     method as payment_method,
#     created_at
# from raw.payments
# """
# n = con.execute(query).fetchall()
# print(n)
# con.close()

[(1, 1, 'bank_transfer', datetime.datetime(2026, 7, 25, 16, 45)), (1, 1, 'card', datetime.datetime(2026, 7, 25, 16, 45)), (2, 2, 'cash_on_delivery', datetime.datetime(2026, 3, 6, 13, 53)), (2, 2, 'cash_on_delivery', datetime.datetime(2026, 3, 6, 13, 53)), (3, 3, 'bank_transfer', datetime.datetime(2026, 5, 30, 14, 36)), (3, 3, 'wallet', datetime.datetime(2026, 5, 30, 14, 36)), (3, 3, 'card', datetime.datetime(2026, 5, 30, 14, 36)), (4, 4, 'bank_transfer', datetime.datetime(2026, 4, 18, 18, 22)), (4, 4, 'cash_on_delivery', datetime.datetime(2026, 4, 18, 18, 22)), (5, 5, 'card', datetime.datetime(2026, 7, 27, 20, 27)), (6, 6, 'card', datetime.datetime(2026, 3, 21, 10, 24)), (6, 6, 'card', datetime.datetime(2026, 3, 21, 10, 24)), (7, 7, 'bank_transfer', datetime.datetime(2026, 5, 10, 3, 35)), (7, 7, 'card', datetime.datetime(2026, 5, 10, 3, 35)), (7, 7, 'card', datetime.datetime(2026, 5, 10, 3, 35)), (8, 8, 'cash_on_delivery', datetime.datetime(2026, 5, 19, 16, 52)), (9, 9, 'bank_transfer'

In [11]:
for t in TABLES:
    con.execute(f"""
                CREATE OR REPLACE TABLE raw.{t} AS
                SELECT *, CURRENT_TIMESTAMP AS _loaded_at
                FROM read_csv('data/{t}.csv', header = true)
                """)
    
    n = con.execute(f"SELECT COUNT(*) FROM raw.{t}").fetchone()[0]
    print(f"raw.{t:<12} {n:>5} rows")
con.close()

raw.customers      200 rows
raw.products         8 rows
raw.orders        1201 rows
raw.order_items   2973 rows
raw.payments      2973 rows


In [13]:
# Quick check
print(duckdb.connect('shopflow.duckdb').sql('select status, count(*) as n from raw.orders group by 1 order by 2 desc'))

┌───────────┬───────┐
│  status   │   n   │
│  varchar  │ int64 │
├───────────┼───────┤
│ DELIVERED │   216 │
│ delivered │   214 │
│ Shipped   │   203 │
│ placed    │   193 │
│ returned  │   193 │
│ shipped   │   182 │
└───────────┴───────┘

